# 03. RQ3, what happens at 96 and 200 training languages

> Reference notebook. Every number below is produced by the same functions that
> write the paper's tables (the analysis `scripts/` folder), so this is a readable
> view of the pipeline rather than a second implementation. Regenerate the
> `.tex` files with `python scripts/run_all.py`.


**Question.** Do the configurations selected on ten languages still lead once
training covers 96 and 200 languages?

Backs Table 2 (final ranking and verdicts), Table 3 (selection transfer),
Table A11 (tests against the base model), and Table A12 (omnibus tests).


In [1]:
import sys
from pathlib import Path

import pandas as pd

# The analysis scripts live in paper_resources/scripts/ in the writing repo and
# in paper_artifact/scripts/ in the code repo. Find whichever is present.
SCRIPTS_DIR = next(
    d
    for p in Path.cwd().resolve().parents
    for d in (p / "paper_resources" / "scripts", p / "paper_artifact" / "scripts")
    if (d / "common.py").is_file()
)
REPO_ROOT = SCRIPTS_DIR.parents[1]
sys.path.insert(0, str(SCRIPTS_DIR))

import common as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
# 4 significant digits rather than 4 decimals, so that a p-value of 5.6e-15 stays
# readable next to an effect size of 0.94.
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

cfg = C.load_config()
BENCHMARKS = cfg["analysis"]["benchmarks"]
ALPHA = cfg["analysis"]["alpha"]
print(f"benchmarks: {BENCHMARKS}")
print(f"alpha     : {ALPHA}")

benchmarks: ['Belebele', 'FLORES', 'SIB200', 'Taxi1500']
alpha     : 0.05


Positions are comparable **within** a regime and not across the two, because
the regimes use different evaluation populations, so the base model's own
numbers differ.


In [2]:
import tab5_rq3_final as rq3

baseline = cfg["rq3"]["baseline_run_id"]
regimes = cfg["rq3"]["regimes"]
panels = {r: C.load_final_panel(C.results_path(cfg, f"final_{r}_xlsx"), cfg) for r in regimes}

for regime, panel in panels.items():
    print(f"{regime:>4} training languages: {panel['language'].nunique():>3} evaluated languages, "
          f"{len(panel):>5} rows, {panel['run_id'].nunique()} systems")

  96 training languages:  80 evaluated languages,  1992 rows, 8 systems
 200 training languages: 136 evaluated languages,  3264 rows, 8 systems


In [3]:
rankings = {r: C.final_ranking(panels[r], BENCHMARKS) for r in regimes}
labels = {cfg["rq3"]["baseline_run_id"]: cfg["rq3"]["baseline_label"]}
labels.update({e["run_id"]: e["label"] for e in cfg["rq3"]["finalists"]})

for regime in regimes:
    print(f"=== {regime} training languages (lower is better)")
    display(rankings[regime].rename(index=labels).round(2))

=== 96 training languages (lower is better)


score_family,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,
MOELoRA r2/32 k8,4.25,3.35,1.49,5.36,3.61
LoRA r2,4.08,4.38,4.19,2.86,3.88
MOELoRA r2/16 k8,4.28,4.11,4.14,3.04,3.89
MTL-LoRA r8/16,5.22,3.76,5.92,3.27,4.54
Base model,2.08,6.87,4,5.24,4.55
HydraLoRA r2,4,5.28,3.59,6.12,4.75
AdaMoE r2/8 k6,6.3,4.23,5.76,4.53,5.21
LoRA r4,5.8,4.02,6.91,5.58,5.58


=== 200 training languages (lower is better)


score_family,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,
HydraLoRA r2,4.35,2.62,1.64,5.18,3.45
LoRA r4,5.71,4.28,5.36,1.41,4.19
Base model,1.79,6.04,3.54,5.59,4.24
MOELoRA r2/32 k8,3.89,4.77,4.01,4.51,4.29
MTL-LoRA r8/16,4.77,5.54,3.44,4.72,4.62
LoRA r2,4.01,3.14,7.78,3.91,4.71
AdaMoE r2/8 k6,6.47,3.48,3.88,5.98,4.95
MOELoRA r2/16 k8,5.01,6.14,6.36,4.7,5.55


## The unadapted model is hard to beat

The base model places fourth of six at 96 training languages and **second of
six** at 200. No configuration is best on more than two of the four benchmarks
in either regime, and none is best in both regimes.


In [4]:
tests = {r: C.final_pairwise_tests(panels[r], BENCHMARKS) for r in regimes}
omnibus = pd.concat(
    {r: tests[r].groupby("benchmark", sort=False).first().reset_index() for r in regimes},
    names=["regime"],
).reset_index(level=0)
omnibus[["regime", "benchmark", "n_units", "friedman_q", "kendalls_w", "p_friedman"]]

,regime,benchmark,n_units,friedman_q,kendalls_w,p_friedman
0,96,Belebele,40,79.7,0.2846,1.587e-14
1,96,FLORES,100,142.7,0.2039,1.373e-27
2,96,SIB200,51,170.9,0.4786,1.642e-33
3,96,Taxi1500,58,118.1,0.2909,1.89e-22
0,200,Belebele,65,148.7,0.3269,7.487e-29
1,200,FLORES,166,350.7,0.3018,8.81e-72
2,200,SIB200,84,363.9,0.6188,1.314e-74
3,200,Taxi1500,93,225.3,0.3461,4.858e-45


Kendall's *W* is worth reading before any ordering. At 200 languages Taxi1500's
evaluation cases barely agree on any ordering at all (*W* of about 0.08), so its
ranking is uninformative rather than a tie.


## The selection stage did not predict either outcome

This is the paper's headline. The configuration ranked first after the
ten-language follow-up finishes third in both broader regimes.


In [5]:
transfer = pd.read_csv(C.output_dir(cfg, "data") / "tab7_rq3_transfer.csv")
display(transfer)
pd.read_csv(C.output_dir(cfg, "data") / "tab7_rq3_transfer_correlations.csv")

,run_id,label,followup_rank_of_22,followup,96,200
0,mtllora_lr5e5_r8_nup16,MTL-LoRA r8/16,1,1,3,3
1,moelora_tr32_e16_k8,MOELoRA r2/16 k8,2,2,2,5
2,hydralora_r2,HydraLoRA r2,3,3,4,1
3,moelora_tr64_e32_k8,MOELoRA r2/32 k8,4,4,1,2
4,mola_tr16_e8_k6,AdaMoE r2/8 k6,5,5,5,4


,pair,spearman_rho
0,followup vs 96,0.3
1,followup vs 200,-0.1
2,96 vs 200,0


Rank-order agreement across stages is essentially absent, with rho of 0.30 from
ten languages to 96, rho of -0.10 from ten to 200, and rho of 0.00 from 96 to
200. With five configurations these are descriptive rather than inferential, but
they match what the tables show.

**The reduced-scale search identified a workable region without identifying the
ordering inside it.**
